# arXiv Nhóm 1 — Crawl & Benchmark

One entrypoint for the M1 CPU-only Colab crawl and benchmark. Final mode covers all four assigned ranges (20,000 IDs), every discovered version, and Semantic Scholar references. Sample mode is for validation only.

Drafted with assistance from OpenAI Codex; follows the group's AI disclosure convention.

**Before Run all:** select CPU/None hardware accelerator. Coordinate one active group crawler. Use the designated Google crawl account for Drive authorization. Create `arxiv-nhom1` in its My Drive first. Do not run a second notebook or local arXiv crawler in parallel.

The first setup initializes a team marker only with explicit permission. New runs never delete old data. After a runtime reset, select RESUME with the same run ID/settings. A four-hour session budget stops between chunks; Colab can terminate earlier. A missing timing summary is disclosed, never silently counted as zero.


:::{note}
This is a generated, nonexecuting mirror of the operational notebook. The Book build never crawls arXiv or mounts Drive. Final measurements are pending.

[Open this version in Colab](https://colab.research.google.com/github/hcmus-arxiv-nhom1/CodeRepo/blob/8b9249ee09e323b1b168f4e045497df412893425/src/notebooks/arxiv_nhom1_crawl_benchmark.ipynb) · [Canonical source](https://github.com/hcmus-arxiv-nhom1/CodeRepo/blob/8b9249ee09e323b1b168f4e045497df412893425/src/notebooks/arxiv_nhom1_crawl_benchmark.ipynb) · Download notebook: {download}`./arxiv_nhom1_crawl_benchmark.ipynb`

CodeRepo notebook revision: `8b9249ee09e323b1b168f4e045497df412893425`. The setup cell separately pins the implementation revision used by the crawler.
:::

In [ ]:
# Edit this cell. Use a NEW run ID for each new experiment.
MODE = "final"  # "sample" for a trial; sample results are never final
RUN_ID = "m1-final-measured-20261031"
RANGES = ("thien", "tien", "hieu", "dang")
OFFSET = 0
COUNT = None  # full ranges; for a sample use MODE="sample", RANGES=("dang",), COUNT=100
CHUNK_SIZE = 100
WITH_REFERENCES = True
CHECKPOINT_S = 300
SESSION_BUDGET_S = 4 * 60 * 60
RESUME = False  # True in a new runtime after a saved run/timeout
SHIFT_OWNER = "dang"  # who is operating the shared crawl account now
TAKEOVER = False  # only after confirming the previous runtime has stopped
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
TEAM_ID = "arxiv-nhom1"
INITIALIZE_TEAM_FOLDER = False  # True once, after checking the intended Drive folder
CODE_REF = "8310f7eb6d244dae5ad244e4d4632604b57efb36"  # immutable implementation revision
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Authorize the fixed Drive and resolve the saved code revision

The designated account owns the Drive destination. Team members operate in shifts. On resume, use a fresh local destination and the saved implementation revision. Corpus and journals are restored together from verified checkpoints; incomplete uploads are ignored.


In [ ]:
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
if not DRIVE.is_dir():
    raise FileNotFoundError("Create/check the intended team Drive folder before running")

revision = CODE_REF
if RESUME:
    saved = json.loads((DRIVE / "runs" / RUN_ID / "MANIFEST.json").read_text())
    revision = saved["code_revision"]

repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True
)
if not Path("/usr/bin/time").is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "time"], check=True)
for name in ("ARXIV_CONTACT_EMAIL", "S2_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value.strip()
    print(name, "configured" if value else "not configured")
# A fresh local directory allows verified restore without deleting a previous copy.
LOCAL_ROOT = Path("/content/m1-local") / uuid.uuid4().hex

## Preflight and exact scope

Check the displayed ID count, code revision, CPU runtime and Drive location before the long-running cell. A 100-ID Dang sample spans 2604.20535–2604.20634. Final mode rejects partial ranges or references disabled.


In [ ]:
from dataclasses import asdict

import psutil

from src.benchmark.checkpoints import prepare_drive
from src.benchmark.runner import BenchmarkConfig, plan_chunks, run_benchmark

prepare_drive(DRIVE, TEAM_ID, initialize=INITIALIZE_TEAM_FOLDER)
cfg = BenchmarkConfig(
    run_id=RUN_ID,
    mode=MODE,
    ranges=tuple(RANGES),
    offset=OFFSET,
    count=COUNT,
    chunk_size=CHUNK_SIZE,
    references=WITH_REFERENCES,
    checkpoint_s=CHECKPOINT_S,
)
chunks = plan_chunks(cfg)
print(asdict(cfg))
print("Selected IDs:", sum(len(c.ids) for c in chunks), "chunks:", len(chunks))
print("CPU count:", psutil.cpu_count(), "RAM MB:", psutil.virtual_memory().total / 1e6)
print("Drive run:", DRIVE / "runs" / RUN_ID)
subprocess.run(["lscpu"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

## Crawl, checkpoint and benchmark

Each chunk uses the existing crawler under the process-tree profiler and `/usr/bin/time -v`. Periodic checkpoints run inside the measured crawler. Post-chunk sync is reported separately. Retry and backoff remain in wall time; do not add them again. Setup/install and offline gaps between sessions are excluded from pipeline time.

Checkpoint target: five minutes at completed-version boundaries. A large version or failed Drive upload can exceed that interval. Only verified Drive checkpoints survive a destroyed runtime. A file being downloaded may need to restart after runtime loss; live HTTP Range recovery is not a cross-runtime promise.

**To end a shift:** normally let the session budget stop the run. If using Stop, wait until processes have stopped and the final checkpoint completes. After an abrupt disconnect, confirm the old runtime is dead before explicit takeover. Drive shift markers are advisory, not atomic distributed locks.


In [ ]:
result = run_benchmark(
    cfg,
    drive_root=DRIVE,
    local_root=LOCAL_ROOT,
    revision=REVISION,
    team_id=TEAM_ID,
    owner=SHIFT_OWNER,
    resume=RESUME,
    takeover=TAKEOVER,
    session_budget_s=SESSION_BUDGET_S,
)
print(json.dumps(result, indent=2))
print("Local corpus for M2:", LOCAL_ROOT / RUN_ID / "data/corpus")
print("Durable checkpoints:", DRIVE / "runs" / RUN_ID / "checkpoints")

## Inspect measurements and download

Full-session RSS includes the runner/notebook kernel baseline and all descendants. Session wall includes restore, crawling, references and final corpus sync. Samples target independent Drive publication every 30 seconds; corpus checkpoints still target 300 seconds. Peaks are sampled observations.

If a runtime dies before finalization, its last durable sample establishes a lower bound. Exact end-to-end and mean-time metrics are omitted until evidence is complete; do not replace missing time with zero. Corpus completeness and timing completeness are separate.

`paper_attempt_timings.csv` retains admission-to-readiness lifecycle latency for each paper/attempt, including batch queue waiting and required references. These latencies overlap. Use session wall / required IDs for amortized mean time. `source_work_s` measures version work separately.

Use a new run ID for this measurement revision. Setup, final evidence re-export/download and offline gaps are excluded. Raw corpus stays in verified Drive checkpoints for M2; the ZIP contains measurements, logs and QA.


In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

from src.benchmark.runner import export_evidence

local = LOCAL_ROOT / RUN_ID
archive = export_evidence(cfg, local)
for name in ("coverage.json", "timing_audit.json"):
    p = local / "work" / name
    if p.exists():
        print(name, p.read_text())
print("Measurement audit:", (local / "book_csv/measurement_audit.json").read_text())
for name in ("benchmark_summary.csv", "corpus_summary.csv"):
    p = local / "book_csv" / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
p = local / "book_csv/paper_attempt_timings.csv"
if p.exists():
    print("Per-paper attempts (first 10; all rows in ZIP)")
    display(pd.read_csv(p).head(10))
print("Evidence:", archive)
files.download(str(archive))

## Resume in a fresh runtime

Keep RUN_ID and all dataset settings unchanged; set RESUME=True. Leave INITIALIZE_TEAM_FOLDER=False. The setup cell loads the saved code revision. Restore validates checkpoint hashes before reuse and skips only chunks passing corpus/journal/reference QA. The newest incomplete upload is ignored in favor of the latest valid chain.

If another shift marker remains after timeout, confirm the previous runtime has actually stopped, then set TAKEOVER=True for this handoff. Do not use another account/runtime to run a second crawler concurrently. The final report records sessions separately and does not claim one uninterrupted runtime.
